# ViFinQA — build reusable full-quality BGE-M3 artifact

Run once with T4 x2, Internet, and the `vifinqa-artifacts` input. This notebook embeds all 146,246 tables with the pinned BGE-M3 revision at its full 8,192-token context in FP32. It uses exact-token length sorting and adaptive batches to remove padding without changing embeddings. It saves resume shards, validates the final FAISS index, and creates `/kaggle/working/vifinqa-dense-bge-m3.zip`. Upload that ZIP as a Kaggle Dataset and reuse it in the generation notebook. To resume across Kaggle sessions, attach the previous notebook output; its checkpoint shards are imported automatically.

In [ ]:
import base64
import hashlib
import json
import os
import shutil
import subprocess
import sys
from pathlib import Path

import torch

try:
    from kaggle_secrets import UserSecretsClient

    hf_token = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    hf_token = None
if hf_token:
    os.environ["HF_TOKEN"] = hf_token
    del hf_token
    print("authenticated Hugging Face downloads enabled")

MODEL = "BAAI/bge-m3"
MODEL_REVISION = "5617a9f61b028005a4858fdac845db406aefb181"
MANIFEST_SHA256 = "ced1d671d6a71c299fea02d7d12b86b596b430a2714ab9aeaa4b338fe012fac1"
assert torch.cuda.is_available(), "Enable a GPU accelerator."
assert torch.cuda.device_count() >= 2, "Select the T4 x2 accelerator for the one-time build."
for index in range(torch.cuda.device_count()):
    properties = torch.cuda.get_device_properties(index)
    print(index, properties.name, round(properties.total_memory / 2**30, 1), "GiB")

In [ ]:
GIT_URL = "https://github.com/ThanhDatVN/AI-Financial-Data-Assistant.git"
PROJECT = Path("/kaggle/working/AI-Financial-Data-Assistant")


def remove_partial_checkout() -> None:
    if PROJECT.exists() and not (PROJECT / "pyproject.toml").exists():
        shutil.rmtree(PROJECT)


def clone_repo() -> None:
    public = subprocess.run(
        ["git", "clone", "--depth", "1", GIT_URL, str(PROJECT)],
        capture_output=True,
        text=True,
    )
    if public.returncode == 0:
        return
    remove_partial_checkout()
    try:
        from kaggle_secrets import UserSecretsClient

        token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    except Exception:
        token = None
    if not token:
        raise RuntimeError("Enable Internet or configure a read-only GITHUB_TOKEN secret.")
    auth = base64.b64encode(f"x-access-token:{token}".encode()).decode()
    environment = os.environ.copy()
    environment.update(
        {
            "GIT_CONFIG_COUNT": "1",
            "GIT_CONFIG_KEY_0": "http.extraHeader",
            "GIT_CONFIG_VALUE_0": f"Authorization: Basic {auth}",
        }
    )
    result = subprocess.run(
        ["git", "clone", "--depth", "1", GIT_URL, str(PROJECT)],
        env=environment,
        capture_output=True,
        text=True,
    )
    del token, auth, environment
    if result.returncode != 0:
        remove_partial_checkout()
        raise RuntimeError("Authenticated clone failed.")


remove_partial_checkout()
attached_candidates = sorted(
    {
        marker.parent
        for marker in Path("/kaggle/input").rglob("pyproject.toml")
        if (marker.parent / "scripts/22_build_dense.py").is_file()
    },
    key=str,
)
if not PROJECT.exists() and attached_candidates:
    shutil.copytree(attached_candidates[0], PROJECT)
elif not PROJECT.exists():
    clone_repo()
assert (PROJECT / "requirements-dense.txt").exists(), "Checkout is missing dense requirements."
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", str(PROJECT / "requirements-dense.txt")],
    check=True,
)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(PROJECT)], check=True)
package_probe = (
    "import importlib.metadata as m; import faiss, sentence_transformers, torch; "
    "print('dense runtime:', 'torch', torch.__version__, "
    "'sentence-transformers', m.version('sentence-transformers'), "
    "'transformers', m.version('transformers')); "
    "assert torch.__version__.startswith('2.10.'); "
    "assert m.version('sentence-transformers') == '5.5.1'; "
    "assert m.version('transformers') == '5.5.3'"
)
subprocess.run([sys.executable, "-c", package_probe], check=True)
os.chdir(PROJECT)
PROJECT_SHA = (
    subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
    if (PROJECT / ".git").exists()
    else "attached-archive-no-git-sha"
)
print("project revision:", PROJECT_SHA)

In [ ]:
INPUT_ROOT = Path("/kaggle/input")
manifest_candidates = sorted(
    (path for path in INPUT_ROOT.rglob("table_manifest.jsonl") if path.parent.name == "processed"),
    key=str,
)
assert manifest_candidates, "Attach the `vifinqa-artifacts` Kaggle Dataset."
MANIFEST = manifest_candidates[0]


def sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


assert sha256(MANIFEST) == MANIFEST_SHA256, "Frozen manifest SHA-256 mismatch."
print("verified manifest:", MANIFEST)

In [ ]:
free_bytes = [torch.cuda.mem_get_info(index)[0] for index in range(torch.cuda.device_count())]
gpu_indices = sorted(
    (index for index, free in enumerate(free_bytes) if free / 2**30 >= 6),
    key=free_bytes.__getitem__,
    reverse=True,
)
assert len(gpu_indices) >= 2, "Both T4 GPUs must have at least 6 GiB free. Restart stale sessions."
devices = [f"cuda:{index}" for index in gpu_indices[:2]]
print("dense devices:", devices)
ARTIFACT_ROOT = Path("/kaggle/working/vifinqa-dense-bge-m3")
DENSE = ARTIFACT_ROOT / "data/index/bge_m3"
CHECKPOINTS = Path("/kaggle/working/dense-checkpoints/bge_m3")
resume_candidates = sorted(
    INPUT_ROOT.rglob("dense-checkpoints/bge_m3/config.json"), key=str
)
if not CHECKPOINTS.exists() and resume_candidates:
    resume_source = resume_candidates[0].parent
    shutil.copytree(resume_source, CHECKPOINTS)
    print("imported prior checkpoint:", resume_source)
if not (DENSE / "index.faiss").exists():
    command = [
        sys.executable,
        "scripts/22_build_dense.py",
        "--manifest",
        str(MANIFEST),
        "--output",
        str(DENSE),
        "--checkpoint-dir",
        str(CHECKPOINTS),
        "--batch-size",
        "16",
        "--max-batch-tokens",
        "8192",
        "--checkpoint-size",
        "256",
        "--max-seq-length",
        "8192",
        "--sort-by-length",
        "--model-revision",
        MODEL_REVISION,
        "--final-run",
    ]
    for device in devices:
        command += ["--device", device]
    subprocess.run(command, check=True)
config = json.loads((DENSE / "config.json").read_text(encoding="utf-8"))
checkpoint_config = json.loads(
    (CHECKPOINTS / "config.json").read_text(encoding="utf-8")
)
assert config == {
    "model_id": MODEL,
    "model_revision": MODEL_REVISION,
    "max_seq_length": 8192,
    "use_fp16": False,
    "tables": 146246,
}
print("full-quality dense build verified:", config)

In [ ]:
relative_files = [
    Path("data/index/bge_m3/config.json"),
    Path("data/index/bge_m3/index.faiss"),
    Path("data/index/bge_m3/records.jsonl"),
]
files = {
    path.as_posix(): {
        "sha256": sha256(ARTIFACT_ROOT / path),
        "bytes": (ARTIFACT_ROOT / path).stat().st_size,
    }
    for path in relative_files
}
artifact_manifest = {
    "format_version": 1,
    "artifact_type": "vifinqa_bge_m3_dense_index",
    "project_revision": PROJECT_SHA,
    "source_manifest_sha256": MANIFEST_SHA256,
    "model": MODEL,
    "model_revision": MODEL_REVISION,
    "max_seq_length": 8192,
    "use_fp16": False,
    "tables": 146246,
    "runtime_versions": checkpoint_config["runtime_versions"],
    "files": files,
}
(ARTIFACT_ROOT / "artifact_manifest.json").write_text(
    json.dumps(artifact_manifest, indent=2) + "\n", encoding="utf-8"
)
archive_path = Path(
    shutil.make_archive(
        "/kaggle/working/vifinqa-dense-bge-m3",
        "zip",
        root_dir=ARTIFACT_ROOT,
    )
)
print(archive_path, archive_path.stat().st_size, sha256(archive_path))
print(json.dumps(artifact_manifest, indent=2))